# Seminar in Natural Language Processing (650.100, 26W)
## Phishing & Pragmatics
### Nils Pudenz, Pavel __

Requirements:

In [ ]:
!pip install -q datasets kagglehub pandas python-dotenv

## Kaggle Dataset

In [ ]:
import os
import glob
import pandas as pd
from dotenv import load_dotenv
from datasets import load_dataset
import kagglehub

load_dotenv()

def inspect_dataset(df, name, text_col, label_col, benign_val, malicious_val, n_samples=5):
    print(f"{'='*70}\nDATASET: {name}\n{'='*70}")
    print(f"Total rows: {len(df):,}")
    print(f"Columns: {list(df.columns)}\n")
    
    # Distribution table
    counts = df[label_col].value_counts(dropna=False)
    pcts = df[label_col].value_counts(normalize=True, dropna=False) * 100
    dist_df = pd.DataFrame({"Count": counts, "Percentage (%)": pcts.round(2)})
    print("Class Distribution:")
    print(dist_df)
    print("-" * 70)
    
    # Helper to print examples
    for tag, val in [("BENIGN", benign_val), ("MALICIOUS", malicious_val)]:
        print(f"\n--- {n_samples} {tag} EXAMPLES (label == {val}) ---")
        subset = df[df[label_col] == val][text_col].dropna()
        samples = subset.sample(min(n_samples, len(subset)), random_state=42)
        for i, text in enumerate(samples, 1):
            clean_text = str(text).replace("\n", " ").strip()
            print(f"[{i}] {clean_text[:250]}{'...' if len(clean_text) > 250 else ''}\n")

In [ ]:
# Downloads and caches the dataset locally
kaggle_path = kagglehub.dataset_download("kuladeep19/phishing-and-legitimate-emails-dataset")
csv_files = glob.glob(os.path.join(kaggle_path, "*.csv"))
df_kaggle = pd.read_csv(csv_files[0])

inspect_dataset(
    df=df_kaggle,
    name="Kaggle: kuladeep19/phishing-and-legitimate-emails-dataset",
    text_col="text",
    label_col="label",
    benign_val=0,
    malicious_val=1,
    n_samples=5
)

# Bonus: Inspect the phishing_type breakdown in the Kaggle dataset
if "phishing_type" in df_kaggle.columns:
    print("Phishing Type Breakdown (Kaggle):")
    print(df_kaggle["phishing_type"].value_counts())

## Hugging Face Dataset

In [ ]:
hf_ds = load_dataset("zefang-liu/phishing-email-dataset", split="train")
df_hf = hf_ds.to_pandas()

inspect_dataset(
    df=df_hf,
    name="Hugging Face: zefang-liu/phishing-email-dataset",
    text_col="Email Text",
    label_col="Email Type",
    benign_val="Safe Email",
    malicious_val="Phishing Email",
    n_samples=5
)